# Component / Signal：同一张图的本地和子进程部署

这个示例使用合成点云和内存执行器，不连接硬件。点云分析暴露 centroid、voxels、metrics；导航演示暴露 path、velocity_command。它演示组合接口，不实现 SLAM 或真实导航算法。

数据边是 Signal，运行单位是 Component。CloudAnalysis 的部署位置可独立配置，其他组件和调用接口保持相同。


In [ ]:
import json
import os
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").exists() and (p / "rsim").is_dir())
ASSETS = ROOT / "assets" / "components"
ASSETS.mkdir(parents=True, exist_ok=True)
os.environ["NOTEBOOK_ASSETS_ROOT"] = os.path.relpath(ROOT / "assets", Path.cwd())
os.environ["NOTEBOOK_NAME"] = "components"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from examples.components import demo
from scipykit.mtp_initializer import subplots, disp, plt


## 在 Notebook 现有事件循环中运行

`demo` 内部通过 `Runtime(snapshot.output, drive, placement=...)` 反向发现所有 producer。分析组件在子进程，数据通过 DDS 描述信息与共享数组返回；其余协程在当前进程。


In [ ]:
local = await demo(use_process=False, frames=8)
process = await demo(use_process=True, frames=8)
assert local["worker"]["pid"] == local["main_pid"]
assert process["worker"]["pid"] != process["main_pid"]
assert local["stopped_on_close"] and process["stopped_on_close"]
for name, report in (("local", local), ("process", process)):
    (ASSETS / f"{name}.json").write_text(json.dumps(report, indent=2))
    print(name, report["worker"], "voxels:", report["voxel_count"])


## 查看可观察的控制结果

Navigation 只发布命令 Signal，CommandMux 仲裁后由 Connect 写入模拟执行器。最后的手动零速覆盖自动控制；退出 Runtime 时进入安全状态。真实底盘需要其硬件侧 provider 独立检查命令有效期。


In [ ]:
fig, ax = subplots(figsize=(6.6, 3))
for name, report in (("local", local), ("process", process)):
    ax.plot(report["commands"], label=name)
ax.set_xlabel("Accepted command index")
ax.set_ylabel("Simulated linear velocity / m s$^{-1}$")
ax.legend()
disp(fig, "commands")
plt.close(fig)


组件定义和 CLI 入口见 [components.py](components.py)，接口与迁移说明见 [架构文档](../docs/architecture.md)。时间对齐使用 Synchronizer；本例 Bundle 仅取各输入的最新快照。
